# 10장 — 🎯 캡스톤: 나만의 챗봇 조련 (시리즈 대단원)

전 장의 코드 조각을 **하나의 파이프라인**으로 잇습니다:

| 단계 | 출처 | 수학 |
|---|---|---|
| A. 사전학습(이어쓰기 기계) | ch0~1 | P(다음 단어\|문맥) |
| B. SFT(시범 새기기) | ch3 | −log₂ q + 마스킹 |
| C. 보상 모델(취향→숫자) | ch4 | −log σ(r_선택−r_탈락) |
| D. DPO(선호 새기기) | ch7 | −log σ(숨은 점수 마진) |
| E. 평가 + 동치 확인 | ch9·4↔7 | 명시적 심판 ↔ 숨은 점수 |

주사위(random) 없이 전부 결정적으로 돌아갑니다. 마지막 셀에서 졸업장이 나옵니다. 🎓

In [ ]:
# 🤖 AI 튜터 — 공부하다 막히면 오른쪽 아래 버튼 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch10')

## A. 사전학습 — 이어쓰기 기계 (ch0~1)

우리의 '베이스 모델' = 다음 단어 빈도표. 요청을 해도 이어 쓸 뿐입니다.

In [ ]:
import math
from collections import Counter, defaultdict

PRETRAIN = [
    '회식 공지 문구 써 줘 라는 요청을 받은 총무의 흔한 실수',
    '회식 공지는 날짜와 장소가 생명이다',
    '다음 주 목요일 저녁 회사 앞 한식당 예약 완료',
]

counts = defaultdict(Counter)
vocab = set()
for line in PRETRAIN:
    words = line.split()
    vocab.update(words)
    for a, b in zip(words, words[1:]):
        counts[a][b] += 1

def generate(prompt, length=8):
    words = prompt.split()
    for _ in range(length):
        nxt = counts.get(words[-1])
        if not nxt:
            break
        words.append(nxt.most_common(1)[0][0])
    return ' '.join(words)

REQUEST = '회식 공지 문구 써 줘'
base_answer = generate(REQUEST)
print('요청      :', REQUEST)
print('🦁 베이스 :', base_answer)
assert base_answer.startswith(REQUEST), 'A단계: 베이스는 이어쓰기 기계여야 한다'

## B. SFT — 시범을 가중치(표)에 새기기 (ch3)

시범 한 줄을 크로스 엔트로피로 채점하고, 답변 부분(마스킹 통과)만 표를 갱신합니다.

In [ ]:
SMOOTH = 0.01

def q_of(prev, nxt):
    c = counts[prev]
    vocab.add(nxt)
    return (c[nxt] + SMOOTH) / (sum(c.values()) + SMOOTH * len(vocab))

DEMO = '요청: 회식 공지 문구 써 줘 답변: 다음 주 목요일 18시30분 회사 앞 한식당에서 회식이 있습니다 참석 여부를 회신해 주세요'.split()
ANSWER_START = DEMO.index('답변:')

def sft_loss():
    pairs = list(zip(DEMO, DEMO[1:]))[ANSWER_START:]
    return sum(-math.log2(q_of(a, b)) for a, b in pairs) / len(pairs)

loss_before = sft_loss()
for _ in range(12):  # 훈련 = 답변 부분 조합만 표에 +1 (마스킹)
    for a, b in list(zip(DEMO, DEMO[1:]))[ANSWER_START:]:
        counts[a][b] += 1
loss_after = sft_loss()

sft_answer = generate('답변:', length=16)
print(f'SFT 손실: {loss_before:.2f} → {loss_after:.2f} 비트')
print('🏫 SFT 후:', sft_answer)
assert loss_after < loss_before * 0.3, 'B단계: SFT 손실이 크게 내려가야 한다'
assert not sft_answer.startswith(REQUEST), 'B단계: 이제 이어쓰기가 아니라 답변이어야 한다'

## C. 보상 모델 — 당신의 취향을 숫자로 (ch4)

SFT 모델의 후보 답변들에 대한 **선호 쌍**(당신이 라벨러라고 상상)으로
Bradley-Terry 보상 점수를 경사하강으로 역산합니다.

In [ ]:
CANDIDATES = {
    'A': '📢 다음 주 목 18:30 회사 앞 한식당. 수요일까지 회신!  (간결형)',
    'B': '안녕하세요 여러분, 다음 주 목요일 저녁… 회신 부탁드립니다.  (정중 상세형)',
    'C': '회식함. 목. 알아서 오셈.  (무성의형)',
}
# 당신(라벨러)의 선택: 간결형 > 상세형 > 무성의형
PREFS = [('A', 'B'), ('A', 'C'), ('B', 'C'), ('A', 'B'), ('B', 'C')]

def sigmoid(x):
    return 1 / (1 + math.exp(-x))

r = {k: 0.0 for k in CANDIDATES}
for _ in range(40):  # BT 경사하강 (ch4): r_선택 ↑, r_탈락 ↓, 크기는 1−p
    for w, l in PREFS:
        p = sigmoid(r[w] - r[l])
        r[w] += 0.4 * (1 - p)
        r[l] -= 0.4 * (1 - p)

print('학습된 보상 점수(취향의 디지털화):')
for k in sorted(r, key=r.get, reverse=True):
    print(f'   {k}: r = {r[k]:+.2f}   {CANDIDATES[k]}')
assert sorted(r, key=r.get, reverse=True) == ['A', 'B', 'C'], 'C단계: 점수 순위가 선호와 일치해야 한다'

## D. DPO — 선호를 정책에 직접 새기기 (ch7)

이번엔 보상 모델을 **쓰지 않고**, 같은 선호 쌍으로 정책을 직접 조련합니다.
참조 모델 π_ref = SFT 직후 분포(상세형을 가장 자주 냄 — SFT 시범이 그랬으니까).

In [ ]:
KEYS = ['A', 'B', 'C']
PI_REF = {'A': 0.25, 'B': 0.60, 'C': 0.15}  # SFT 모델: 정중 상세형(B)을 주로 냄
z = {k: 0.0 for k in KEYS}

def policy():
    w = {k: PI_REF[k] * math.exp(z[k]) for k in KEYS}
    s = sum(w.values())
    return {k: v / s for k, v in w.items()}

def hidden(k, pi):
    """숨은 점수 r̂ = log(π/π_ref) — 원본보다 얼마나 자주 내는가 (β=1)."""
    return math.log(pi[k] / PI_REF[k])

for _ in range(60):  # DPO: 선호 쌍마다 마진이 커지도록 (걸음폭 1−σ(마진))
    pi = policy()
    for w_key, l_key in PREFS:
        g = 1 - sigmoid(hidden(w_key, pi) - hidden(l_key, pi))
        z[w_key] += 0.3 * g
        z[l_key] -= 0.3 * g

pi = policy()
best = max(pi, key=pi.get)
print('조련된 정책:', {k: f'{v:.0%}' for k, v in pi.items()})
print(f'🐕 당신의 챗봇: {CANDIDATES[best]}')
assert best == 'A', 'D단계: 정책이 당신의 선호(간결형)로 수렴해야 한다'

## E. 마지막 확인 — 두 심판은 같은 방향을 본다 (4장 ↔ 7장)

C단계의 **명시적 보상 점수**(따로 훈련한 심판)와 D단계의 **숨은 점수**(정책 안에서 태어난 심판) —
이론상 같은 것의 두 얼굴이라 했죠(7장). 정말 같은 방향인지 확인합니다.

In [ ]:
print('후보  명시적 심판 r (4장)   숨은 점수 r̂ (7장)')
for k in KEYS:
    print(f'  {k}      {r[k]:+.2f}              {hidden(k, pi):+.2f}')

# 검증 — 모든 쌍에서 두 점수의 우열 방향이 일치
for i in range(len(KEYS)):
    for j in range(i + 1, len(KEYS)):
        a, b = KEYS[i], KEYS[j]
        explicit = r[a] - r[b]
        implicit = hidden(a, pi) - hidden(b, pi)
        assert explicit * implicit > 0, f'{a} vs {b}: 두 심판의 방향이 일치해야 한다!'
print('\n✔ 확인: 명시적 심판과 숨은 점수가 모든 쌍에서 같은 방향 — RLHF와 DPO는 동전의 양면')

In [ ]:
print('=' * 56)
print('  🎓 졸업장 — 수포자를 위한 AI 수학 시리즈')
print('=' * 56)
print('  귀하는 이어쓰기 기계에서 출발하여')
print('  SFT(크로스 엔트로피) → 보상 모델(시그모이드)')
print('  → DPO(숨은 점수, KL 목줄 내장)의 전 과정을')
print('  직접 조립·실행·검증하였기에 이 졸업장을 드립니다.')
print()
print('  ✔ 이어쓰기 → 답변      (SFT: −log₂ q, 마스킹)')
print('  ✔ 비교 → 점수          (BT: −log σ(Δr))')
print('  ✔ 선호 → 정책          (DPO: 숨은 점수 마진)')
print('  ✔ 두 심판의 동치 확인   (4장 ↔ 7장)')
print()
print('  GPT가 ChatGPT가 되는 길은,')
print('  아는 수학 몇 조각의 재조립이었다.')
print('=' * 56)

## 여정의 끝에서

- 이 노트북에는 새 수학이 한 줄도 없습니다 — ch0~9의 조각을 이었을 뿐. 그게 캡스톤의 메시지예요.
- 진짜 파이프라인과의 거리: 표 대신 신경망, 답변 3개 대신 토큰 시퀀스, +1 대신 역전파 —
  하지만 **채점기(−log)·취향의 확률화(σ)·목줄(KL)·엔진(경사하강)** 이라는 뼈대는 동일합니다.
- 다음에 새로운 조련 논문을 만나면 이 네 조각이 어디에 숨어 있는지 찾아보세요 — 대부분 있습니다.

**시리즈 완주를 축하합니다.** 🎉 1부 후반 심화(고유값·SVD·PCA)와 3부 심화(검정·회귀)가
"선택"으로 남아 있어요 — 이제 전부 아는 수학의 이웃입니다.

> 💡 마지막까지: `from tutor import explain` 후 `explain('ch10')` — 어느 셀이든 튜터가 풀어 드립니다.